# 71. Simplify Path

**Difficulty**: Medium  
**Topics**: String, Stack  
**Link**: [LeetCode 71](https://leetcode.com/problems/simplify-path/)

---

## Problem Statement

Given a string `path`, which is an absolute path (starting with a slash `'/'`) to a file or directory in a Unix-style file system, convert it to the simplified canonical path.

In a Unix-style file system, a period `'.'` refers to the current directory, a double period `'..'` refers to the directory up a level, and any multiple consecutive slashes (i.e., `'//'`) are treated as a single slash. For this problem, any other format of periods such as `'...'` are treated as file/directory names.

The canonical path should have the following format:

- The path starts with a single slash `'/'`.
- The path does not end with a trailing `'/'` unless the path is just `"/"`.
- The path does not contain multiple consecutive `'/'`.
- The path does not contain `.` or `..`.

Return the canonical path.

### Examples

**Example 1:**
```
Input: path = "/home/"
Output: "/home"
Explanation: Note that there is no trailing slash after the last directory name.
```

**Example 2:**
```
Input: path = "/../"
Output: "/"
Explanation: Going one level up from the root directory is not possible.
```

**Example 3:**
```
Input: path = "/home//foo/"
Output: "/home/foo"
Explanation: Multiple consecutive slashes are replaced by a single one.
```

### Constraints

- `1 <= path.length <= 3000`
- `path` consists of English letters, digits, period `'.'`, slash `'/'` or `'_'`.

---

## Approach 1: Stack

### Intuition

Use a stack to build the path by processing each component:
- Normal directory: push to stack
- `.`: ignore (current directory)
- `..`: pop from stack if not empty (go up one level)
- Empty string (from `//`): ignore

### Why it works

The stack naturally maintains the directory hierarchy. When we encounter `..`, we need to remove the most recent directory, which is exactly what a stack's pop operation does.

### Algorithm Visualization

```
path = "/a/./b/../../c/"

Split: ["a", ".", "b", "..", "..", "c"]

Stack: []

"a": push -> ["a"]
".": ignore -> ["a"]
"b": push -> ["a", "b"]
"..": pop -> ["a"]
"..": pop -> []
"c": push -> ["c"]

Result: "/c"
```

### Complexity
- **Time**: O(n) - one pass through the path
- **Space**: O(n) - worst case stack size

In [ ]:
def simplify_path_stack(path: str) -> str:
    """Stack approach: O(n) time, O(n) space"""
    stack = []
    
    # Split path by '/' and process each component
    for component in path.split('/'):
        if component == '' or component == '.':
            # Empty string (from //) or current directory - ignore
            continue
        elif component == '..':
            # Parent directory - pop if not empty
            if stack:
                stack.pop()
        else:
            # Normal directory - push to stack
            stack.append(component)
    
    # Build the canonical path
    return '/' + '/'.join(stack)

# Test cases
test_cases = [
    ("/home/", "/home"),
    ("/../", "/"),
    ("/home//foo/", "/home/foo"),
    ("/a/./b/../../c/", "/c"),
    ("/a/../../b/../c//.//", "/c"),
    ("/a//b////c/d//././/..", "/a/b/c"),
    ("/..", "/"),
    ("/.", "/"),
    ("/", "/"),
    ("/...", "/..."),  # ... is a valid directory name
    ("/.hidden", "/.hidden")
]

for path, expected in test_cases:
    result = simplify_path_stack(path)
    print(f"{path!r:25} -> {result!r:15} (expected {expected!r})")

---

## Approach 2: Two Pointers

### Intuition

Process the path using two pointers without splitting the entire string.

### Why it works

We can identify directory boundaries by finding the next slash, avoiding the overhead of string splitting.

### Complexity
- **Time**: O(n)
- **Space**: O(n) - still need stack for path components

In [ ]:
def simplify_path_two_pointers(path: str) -> str:
    """Two pointers approach: O(n) time, O(n) space"""
    stack = []
    i = 0
    n = len(path)
    
    while i < n:
        # Skip consecutive slashes
        while i < n and path[i] == '/':
            i += 1
        
        if i >= n:
            break
        
        # Find the next component
        start = i
        while i < n and path[i] != '/':
            i += 1
        
        component = path[start:i]
        
        if component == '..':
            if stack:
                stack.pop()
        elif component != '.' and component != '':
            stack.append(component)
    
    return '/' + '/'.join(stack)

# Test
for path, expected in test_cases:
    result = simplify_path_two_pointers(path)
    print(f"{path!r:25} -> {result!r:15} (expected {expected!r})")

---

## Approach 3: In-place Processing

### Intuition

Use the path string itself as storage to build the result.

### Why it works

We can treat the path as a character array and overwrite it with the canonical path.

### Complexity
- **Time**: O(n)
- **Space**: O(n) - need to store components, but can optimize further

In [ ]:
def simplify_path_inplace(path: str) -> str:
    """In-place style processing: O(n) time, O(n) space"""
    components = []
    
    # Parse path into components
    i = 0
    while i < len(path):
        # Skip slashes
        while i < len(path) and path[i] == '/':
            i += 1
        
        if i >= len(path):
            break
        
        # Extract component
        start = i
        while i < len(path) and path[i] != '/':
            i += 1
        
        component = path[start:i]
        
        if component == '..':
            if components:
                components.pop()
        elif component != '.' and component != '':
            components.append(component)
    
    return '/' + '/'.join(components)

# Test
for path, expected in test_cases:
    result = simplify_path_inplace(path)
    print(f"{path!r:25} -> {result!r:15} (expected {expected!r})")

---

## Walkthrough (Stack Approach)

Input: `"/a/./b/../../c/"`

| Step | Component | Action | Stack | Reason |
|------|-----------|--------|-------|--------|
| 0 | - | Initialize | [] | Start empty |
| 1 | `""` | Ignore | [] | From leading slash |
| 2 | `"a"` | Push | ["a"] | Normal directory |
| 3 | `"."` | Ignore | ["a"] | Current directory |
| 4 | `"b"` | Push | ["a", "b"] | Normal directory |
| 5 | `".."` | Pop | ["a"] | Go up one level |
| 6 | `".."` | Pop | [] | Go up one level |
| 7 | `"c"` | Push | ["c"] | Normal directory |
| 8 | `""` | Ignore | ["c"] | From trailing slash |

**Final Result**: `"/c"`

In [ ]:
def simplify_path_verbose(path: str) -> str:
    """Verbose stack approach showing steps"""
    stack = []
    
    print(f"Processing path: {path!r}")
    print(f"Split components: {path.split('/')}")
    print()
    
    for i, component in enumerate(path.split('/')):
        print(f"Step {i+1}: Component = {component!r}")
        print(f"  Stack before: {stack}")
        
        if component == '' or component == '.':
            print(f"  Action: Ignore (empty or current directory)")
        elif component == '..':
            if stack:
                popped = stack.pop()
                print(f"  Action: Go up one level, removed {popped!r}")
            else:
                print(f"  Action: At root, cannot go up")
        else:
            stack.append(component)
            print(f"  Action: Add directory {component!r}")
        
        print(f"  Stack after: {stack}")
        print()
    
    result = '/' + '/'.join(stack)
    print(f"Final result: {result!r}")
    return result

# Test with verbose output
simplify_path_verbose("/a/./b/../../c/")

---

## Comparison

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Stack** | O(n) | O(n) | Simple, intuitive, Pythonic | String splitting overhead |
| **Two Pointers** | O(n) | O(n) | No splitting overhead | More complex logic |
| **In-place** | O(n) | O(n) | Memory efficient | Still needs component storage |

**Best choice**: **Stack** - clean and readable for this problem size.

---

## Edge Cases

| Case | Input | Output | Reason |
|------|-------|--------|--------|
| **Root only** | "/" | "/" | Already canonical |
| **Multiple slashes** | "////" | "/" | All collapse to one |
| **Go up from root** | "/../../.." | "/" | Cannot go above root |
| **Current directory** | "/././." | "/" | All ignored |
| **Mixed with dots** | "/a/../b/../c" | "/c" | Go up then down |
| **Trailing slash** | "/home/" | "/home" | Remove trailing |
| **Dot as name** | "/.../" | "/..." | Not a special case |
| **Hidden files** | "/.hidden/" | "/.hidden" | Hidden file is valid |
| **Complex** | "/a//b////c/d//././/.." | "/a/b/c" | Multiple operations |

---

## Common Mistakes

1. **Not handling empty components**
   ```python
   # WRONG: Empty components from // cause issues
   for component in path.split('/'):
       if component == '..':
   # CORRECT: Handle empty components
   for component in path.split('/'):
       if component == '' or component == '.':
           continue
   ```

2. **Going above root**
   ```python
   # WRONG: IndexError when stack is empty
   if component == '..':
       stack.pop()
   # CORRECT: Check if stack is empty
   if component == '..':
       if stack:
           stack.pop()
   ```

3. **Treating dots as special cases**
   ```python
   # WRONG: '...' is not a special case
   if component.startswith('.'):
       # This would incorrectly handle '...'
   # CORRECT: Only '.' and '..' are special
   if component == '.' or component == '..':
   ```

---

## Related Problems

| Problem | Difficulty | Pattern |
|---------|------------|--------|
| [20. Valid Parentheses](https://leetcode.com/problems/valid-parentheses/) | Easy | Stack for validation |
| [224. Basic Calculator](https://leetcode.com/problems/basic-calculator/) | Hard | Stack for expression evaluation |
| [388. Longest Absolute File Path](https://leetcode.com/problems/longest-absolute-file-path/) | Medium | Stack for directory traversal |
| [150. Evaluate Reverse Polish Notation](https://leetcode.com/problems/evaluate-reverse-polish-notation/) | Medium | Stack for evaluation |

---

## Key Takeaways

| Pattern | When to Use |
|---------|-------------|
| **Stack for path building** | When processing hierarchical paths |
| **Split and process** | When path has clear separators |
| **Handle edge cases** | Empty strings, root directory, going up |
| **Special characters** | Only '.' and '..' are special in Unix paths |
| **Canonical form** | Always start with '/', no trailing '/', no consecutive '/' |

**Remember**: Stack is perfect for path operations where you need to go back and forth between directory levels!